# 07 · Soft sub-lever Sweep — blend 세기 3-D 튜닝 (EXP #33, LB 0.6914)

직전 = #32 Soft(raw p_clf, LB 0.6910). Soft는 sub-lever space의 단일 점(T=1, no cap, α=1)일 뿐이었다.

**가정**: 약한 classifier(recovery-AUC 0.5547) 환경에선 p_clf 분포가 0.5 근처에 압축돼 있다. 이를 **T(temperature soften)·max_w(cap)·α(centered shrinkage)** 로 조절하면 균등 blend에 가깝게 밀어 minority boundary를 추가로 회수할 수 있다(특히 T>1 soften / α<1).

**실험**: σ T × max_w Δ × α = 3×3×3 = 27 cell full grid를 cache 재사용으로 학습 0 평가. 27 cell sweep이라 G1 strict는 통과 불가 → **Floor(baseline 초과) + 4th guard primary** framework. best 1개만 submission(memory 정책).

**결과**: **T=1.5 soften 단독 dominant**(max_w cap dead, α<1은 T>1과 중복) → **LB 0.6914 (+0.0004)**. #32+#33 cascade 누적 +0.0008. T가 "선택"이 아니라 "blend 세기"만 조절함을 확인(sub-noise zone — prior 갱신 신중).

## 셀 0 — imports + 상수 + 3-D grid 정의

In [ ]:
import os, time, json, itertools
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01
SEEDS = [42, 7, 123]

# Classifier hyperparam (LGBM binary, #31/#32 동일) — Phase B에서만 사용
clf_params = dict(
    objective='binary', metric='binary_logloss',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=10,
    max_bin=511, n_estimators=300, random_state=42, verbose=-1,
)

# K15/GRU reference HR (#32 셀 5/6 정량)
K15_REF_HR = dict(overall=0.6649, major=0.7312, minor=0.3105)
GRU_REF_OVERALL = 0.6659
GRU_REF_MINOR = 0.3181
ORACLE_KG_HR = dict(overall=0.6870, major=0.7474, minor=0.3638)
CEILING_GAIN_KG = dict(overall=0.0221, major=0.0162, minor=0.0533)

# Soft baseline (#32 실측 OOF — Floor primary)
SOFT_BASELINE_HR_OVERALL = 0.6686
SOFT_BASELINE_DO = 0.0037           # Floor primary 임계
SOFT_BASELINE_DM_REF = 0.0127       # #32 실측 minor delta (참고)
MINORGRU_FLOOR_DO = 0.0012          # #32 정량 (informational reference)

# 3-D sub-lever grid (plan §4.18 권장값)
SIGMA_T_GRID    = [0.7, 1.0, 1.5]         # T<1 sharpen / T=1 base / T>1 soften
MAX_W_DELTA_GRID = [0.10, 0.20, float('inf')]  # Δ=∞ no cap (base)
ALPHA_GRID      = [0.5, 0.75, 1.0]        # α=1 base, α<1 K15 끌어당김
BASELINE_CELL = (1.0, float('inf'), 1.0)  # Soft-base (#32)
N_CELLS = len(SIGMA_T_GRID) * len(MAX_W_DELTA_GRID) * len(ALPHA_GRID)
assert N_CELLS == 27

# Phase A 가드 — G1 informational + Floor primary
COMBINED_STD = 0.0014
G1_INFO_THR  = COMBINED_STD * (N_CELLS ** 0.5)   # ≈ 0.0073 informational only
G4_THR       = COMBINED_STD * (2 ** 0.5)         # ≈ 0.0020 primary per-cell
FLOOR_PRIMARY = SOFT_BASELINE_DO                   # ≈ 0.0037 primary baseline 초과

# Phase B sanity (조건부 classifier 강화)
AUC_RECOVERY_STRONG_THR = 0.60   # Phase B strict (recovery-AUC 강화 임계)
AUC_OVERALL_SANITY = 0.50
FOLD_STD_MAX = 0.020

os.makedirs('results', exist_ok=True)
print(f'SEEDS={SEEDS}')
print(f'3-D Grid: σ T {SIGMA_T_GRID} × max_w Δ {MAX_W_DELTA_GRID} × α {ALPHA_GRID} = {N_CELLS} cell')
print(f'Baseline cell (#32 Soft-base): T={BASELINE_CELL[0]}, Δ={BASELINE_CELL[1]}, α={BASELINE_CELL[2]}')
print(f'G1 informational: +{G1_INFO_THR:.4f} (√{N_CELLS}·{COMBINED_STD:.4f})')
print(f'G4 primary: +{G4_THR:.4f} (per-cell LB자격)')
print(f'Floor primary: +{FLOOR_PRIMARY:.4f} (Soft baseline 초과, OOF overfit 차단)')
print(f'K15 ref HR overall={K15_REF_HR["overall"]:.4f}, Soft baseline OOF={SOFT_BASELINE_HR_OVERALL:.4f} (+{SOFT_BASELINE_DO:.4f})')


## 셀 1 — helpers + `apply_soft_lever` 3-축 변환 함수

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def make_splits(stratify_int, seed, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(stratify_int)), stratify_int)]


def hr_triple(hit_arr, minority_mask):
    return dict(
        overall=float(hit_arr.mean()),
        major=float(hit_arr[~minority_mask].mean()),
        minor=float(hit_arr[minority_mask].mean()),
    )


def apply_soft_lever(p, T=1.0, Delta=float('inf'), alpha=1.0):
    """3-축 sub-lever 변환: p → σ scaling → max_w clip → centered shrinkage → w.

    Steps:
    1. σ (temperature T): w_T = sigmoid(logit(p) / T)  [T==1.0 → w_T = p]
    2. max_w (cap Δ): w_D = clip(w_T, 0.5−Δ, 0.5+Δ)  [Δ==inf → w_D = w_T]
    3. centered shrinkage α: w' = 0.5 + α (w_D − 0.5)  [α==1.0 → w' = w_D]

    Baseline (T=1, Δ=∞, α=1) → w' = p (identity).
    """
    if T == 1.0:
        w = p.astype(np.float32).copy()
    else:
        eps = 1e-6
        p_clip = np.clip(p, eps, 1 - eps)
        logit = np.log(p_clip / (1 - p_clip))
        w = (1.0 / (1.0 + np.exp(-logit / T))).astype(np.float32)
    if Delta < float('inf'):
        w = np.clip(w, 0.5 - Delta, 0.5 + Delta)
    if alpha != 1.0:
        w = 0.5 + alpha * (w - 0.5)
    return w.astype(np.float32)


def cell_name(T, Delta, alpha):
    """Cell 명명 encoding: T{T*10}_D{Δ*100|INF}_a{α*100:03d}."""
    t_s = f'T{int(round(T*10)):02d}'
    if Delta == float('inf'):
        d_s = 'DINF'
    else:
        d_s = f'D{int(round(Delta*100)):02d}'
    a_s = f'a{int(round(alpha*100)):03d}'
    return f'{t_s}_{d_s}_{a_s}'


def run_classifier_seeds(X_tr, y_target, X_ts, stratify, seeds=SEEDS):
    """Phase B 전용. 3 seed × 5 fold = 15 model."""
    N = len(y_target)
    oof_probs = []
    test_probs = [] if X_ts is not None else None
    per_seed = []
    for seed in seeds:
        folds = make_splits(stratify, seed=seed)
        oof_prob = np.full(N, np.nan, dtype=np.float32)
        test_prob_seed = np.zeros(len(X_ts), dtype=np.float32) if X_ts is not None else None
        fold_accs = []
        t0 = time.time()
        for tr_idx, val_idx in folds:
            m = lgb.LGBMClassifier(**clf_params)
            m.fit(X_tr[tr_idx], y_target[tr_idx],
                  eval_set=[(X_tr[val_idx], y_target[val_idx])],
                  callbacks=[lgb.early_stopping(30, verbose=False)])
            p_val = m.predict_proba(X_tr[val_idx])[:, 1].astype(np.float32)
            oof_prob[val_idx] = p_val
            if X_ts is not None:
                test_prob_seed += m.predict_proba(X_ts)[:, 1].astype(np.float32) / N_FOLDS
            fold_accs.append(accuracy_score(y_target[val_idx], (p_val > 0.5).astype(int)))
        oof_probs.append(oof_prob)
        if X_ts is not None:
            test_probs.append(test_prob_seed)
        acc_seed = accuracy_score(y_target, (oof_prob > 0.5).astype(int))
        auc_seed = roc_auc_score(y_target, oof_prob)
        fold_std = float(np.std(fold_accs, ddof=1))
        per_seed.append(dict(seed=seed, accuracy=float(acc_seed), auc=float(auc_seed),
                              fold_accs=[float(a) for a in fold_accs],
                              fold_std=fold_std, time_sec=time.time() - t0))
    oof_prob_mean = np.mean(oof_probs, axis=0).astype(np.float32)
    test_prob_mean = np.mean(test_probs, axis=0).astype(np.float32) if X_ts is not None else None
    return dict(oof_prob_mean=oof_prob_mean, test_prob_mean=test_prob_mean,
                per_seed=per_seed, oof_probs_per_seed=oof_probs,
                test_probs_per_seed=test_probs)


def compute_classifier_metrics(oof_prob_mean, y_target, recovery_mask=None, per_seed=None):
    overall_acc = float(accuracy_score(y_target, (oof_prob_mean > 0.5).astype(int)))
    overall_auc = float(roc_auc_score(y_target, oof_prob_mean))
    if recovery_mask is not None and recovery_mask.sum() > 0:
        rec_auc = float(roc_auc_score(y_target[recovery_mask], oof_prob_mean[recovery_mask]))
    else:
        rec_auc = float('nan')
    fold_std_mean = float(np.mean([m['fold_std'] for m in per_seed])) if per_seed else float('nan')
    return dict(overall_acc=overall_acc, overall_auc=overall_auc,
                recovery_auc=rec_auc, fold_std_mean=fold_std_mean)


# Sanity: baseline cell apply_soft_lever identity check
_p_test = np.array([0.1, 0.3, 0.5, 0.7, 0.9], dtype=np.float32)
_w_base = apply_soft_lever(_p_test, *BASELINE_CELL)
assert np.allclose(_w_base, _p_test, atol=1e-6), f'baseline cell apply_soft_lever non-identity: {_w_base} vs {_p_test}'
print('apply_soft_lever baseline identity OK')
print(f'  baseline cell name: {cell_name(*BASELINE_CELL)}')

# Print all 27 cell names
print(f'\n27 cell names:')
_all_cells = list(itertools.product(SIGMA_T_GRID, MAX_W_DELTA_GRID, ALPHA_GRID))
for i, (T, D, a) in enumerate(_all_cells):
    flag = ' (baseline)' if (T, D, a) == BASELINE_CELL else ''
    print(f'  [{i+1:2d}] {cell_name(T, D, a):<18}  T={T}  Δ={D}  α={a}{flag}')


## 셀 2 — Drive mount + 데이터 + cache 동기화

**필수 cache 18개**:
- K15 OOF/test (6): EXP #29 `topk_e29_{oof,test}_K15_seed*.npy`
- GRU OOF/test (6): EXP #25 `gru_e25_fren_{oof,test}_seed*.npy`
- **#32 classifier OOF/test (6)**: `split_e32_clf_{oof,test}_seed*.npy` — Phase A 학습 0의 핵심

cache 누락 시 lever 폐기 (재학습 안 함).


In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

K15_OOF_PATHS    = [f'results/topk_e29_oof_K15_seed{s}.npy' for s in SEEDS]
K15_TEST_PATHS   = [f'results/topk_e29_test_K15_seed{s}.npy' for s in SEEDS]
GRU_OOF_PATHS    = [f'results/gru_e25_fren_oof_seed{s}.npy' for s in SEEDS]
GRU_TEST_PATHS   = [f'results/gru_e25_fren_test_seed{s}.npy' for s in SEEDS]
CLF32_OOF_PATHS  = [f'results/split_e32_clf_oof_seed{s}.npy' for s in SEEDS]
CLF32_TEST_PATHS = [f'results/split_e32_clf_test_seed{s}.npy' for s in SEEDS]
all_required = (K15_OOF_PATHS + K15_TEST_PATHS + GRU_OOF_PATHS + GRU_TEST_PATHS
                + CLF32_OOF_PATHS + CLF32_TEST_PATHS)
assert len(all_required) == 18

need_mount = (
    not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR))
    or any(not os.path.exists(p) for p in all_required)
)

if need_mount:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    if not os.path.exists('/content/open'):
        !unzip -q -o "{ZIP_PATH}" -d /content/

for p in all_required:
    if not os.path.exists(p):
        drive_p = f'/content/drive/MyDrive/{p}'
        if os.path.exists(drive_p):
            !cp {drive_p} {p}
missing = [p for p in all_required if not os.path.exists(p)]
assert not missing, f'필수 cache 누락 (K15/GRU/#32 classifier): {missing}'
print(f'K15 + GRU + #32 classifier cache 18/18 OK')


def _resolve_data_dir():
    for cand in ['/content/open', '/content']:
        if os.path.exists(f'{cand}/train_labels.csv'):
            return cand
    return None

DATA_DIR = _resolve_data_dir()


def _resolve_sample_sub(data_dir):
    for p in [f'{data_dir}/sample_submission.csv', f'{data_dir}/submission/sample_submission.csv']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'sample_submission.csv not found under {data_dir}')


if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)

sample_sub_path = _resolve_sample_sub(DATA_DIR)
sample_sub = pd.read_csv(sample_sub_path)
test_ids = sample_sub['id'].tolist()
if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape}, cache 18/18 OK')


## 셀 3 — base + Frenet frame + minority mask

In [ ]:
def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_p = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    n1 = np.linalg.norm(np.cross(eL, z_up), axis=1, keepdims=True)
    eN_fb1 = np.cross(eL, z_up) / (n1 + 1e-9)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1)
    eN = np.where(apn < fb_thresh, eN_fb, eN_p)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32)


def compute_kinematics(traj):
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]
    a_last = a[:, -1, :]
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), jerk_last.astype(np.float32), a_sm.astype(np.float32)


base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
vl_tr, al_tr, jl_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, jl_ts, asm_ts = compute_kinematics(traj_test)
eL_tr, eN_tr, eB_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts = build_frenet_batch(vl_ts, asm_ts)

acc_norm_last_tr = _norm(al_tr)
acc_norm_last_ts = _norm(al_ts)
minority_mask_tr = acc_norm_last_tr >= MINORITY_THRESHOLD
minority_mask_ts = acc_norm_last_ts >= MINORITY_THRESHOLD
print(f'minority train {minority_mask_tr.sum()}/{len(minority_mask_tr)} ({minority_mask_tr.mean():.4f})')
print(f'minority test  {minority_mask_ts.sum()}/{len(minority_mask_ts)} ({minority_mask_ts.mean():.4f})')


## 셀 4 — V3 15 + V3 26 feature builders (Phase B 조건부 사용)

In [ ]:
def build_v3_15(traj, eL, eN, eB):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w_sm = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w_sm[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    a_tang_last = (v_last * a_last).sum(1) / (speed_last + 1e-9)
    a_cent_last = _norm(np.cross(v_last, a_last)) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    vy_std = v[:, :, 1].std(axis=1)
    j_L = (jerk_last * eL).sum(1)
    j_N = (jerk_last * eN).sum(1)
    X = np.column_stack([
        a_tang_last, va_dot, speed_last, j_L, cos_turn_last,
        j_N, a_cent_last, speed_diff_half, turn_mean_half_diff, acc_norm_last,
        radial_v, turn_mean, acc_norm_w3, distance_r, vy_std,
    ]).astype(np.float32)
    return X


def build_v3_26(traj, eL, eN, eB):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w_sm = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w_sm[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    v_std, a_std = v.std(axis=1), a.std(axis=1)
    seg = _norm(p[:, 1:, :] - p[:, :-1, :])
    path_eff = _norm(p[:, -1, :] - p[:, 0, :]) / (seg.sum(1) + 1e-9)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    a_tang_last = (v_last * a_last).sum(1) / (speed_last + 1e-9)
    a_cent_last = _norm(np.cross(v_last, a_last)) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    a_N = (a_last * eN).sum(1); a_B = (a_last * eB).sum(1)
    j_L = (jerk_last * eL).sum(1); j_N = (jerk_last * eN).sum(1); j_B = (jerk_last * eB).sum(1)
    aw_L = (a_w3 * eL).sum(1); aw_N = (a_w3 * eN).sum(1)
    X = np.column_stack([
        speed_last, acc_norm_last, acc_norm_w3,
        v_std[:,0], v_std[:,1], v_std[:,2], a_std[:,0], a_std[:,1], a_std[:,2],
        path_eff, distance_r, radial_v, turn_mean, cos_turn_last, va_dot,
        a_tang_last, a_cent_last, speed_diff_half, turn_mean_half_diff,
        a_N, a_B, j_L, j_N, j_B, aw_L, aw_N,
    ]).astype(np.float32)
    return X


X_tr_v15 = build_v3_15(traj_train, eL_tr, eN_tr, eB_tr)
X_ts_v15 = build_v3_15(traj_test,  eL_ts, eN_ts, eB_ts)
X_tr_v26 = build_v3_26(traj_train, eL_tr, eN_tr, eB_tr)
X_ts_v26 = build_v3_26(traj_test,  eL_ts, eN_ts, eB_ts)
print(f'V3 15: tr {X_tr_v15.shape}  V3 26: tr {X_tr_v26.shape}  (Phase B 조건부 사용)')


## 셀 5 — K15 + GRU cache 로드 + HR sanity + GRU form auto-detect (#32 셀 5 동일)

In [ ]:
def load_seed_avg(paths):
    return np.mean([np.load(p) for p in paths], axis=0).astype(np.float32)


oof_K15 = load_seed_avg(K15_OOF_PATHS)
test_K15 = load_seed_avg(K15_TEST_PATHS)
assert oof_K15.shape == (10000, 3) and test_K15.shape == (10000, 3)

pred_K15_tr = base_train + oof_K15
pred_K15_ts = base_test + test_K15
err_K15 = np.linalg.norm(y_train - pred_K15_tr, axis=1)
hit_K15 = err_K15 < HIT_RADIUS
HR_K15 = hr_triple(hit_K15, minority_mask_tr)
print(f'K15 HR: overall={HR_K15["overall"]:.4f}  major={HR_K15["major"]:.4f}  minor={HR_K15["minor"]:.4f}')

for k in ['overall', 'major', 'minor']:
    diff = abs(HR_K15[k] - K15_REF_HR[k])
    assert diff < 0.0005, f'K15 {k} drift: ref {K15_REF_HR[k]:.4f} vs got {HR_K15[k]:.4f}'
print('K15 sanity OK')

oof_GRU = load_seed_avg(GRU_OOF_PATHS)
test_GRU = load_seed_avg(GRU_TEST_PATHS)

pred_cart = base_train + oof_GRU
err_cart = np.linalg.norm(y_train - pred_cart, axis=1)
HR_cart_overall = float((err_cart < HIT_RADIUS).mean())

oof_GRU_inv = (oof_GRU[:, 0:1] * eL_tr + oof_GRU[:, 1:2] * eN_tr + oof_GRU[:, 2:3] * eB_tr)
pred_fren = base_train + oof_GRU_inv
err_fren = np.linalg.norm(y_train - pred_fren, axis=1)
HR_fren_overall = float((err_fren < HIT_RADIUS).mean())

print(f'\nGRU form auto-detect:')
print(f'  Cart    HR = {HR_cart_overall:.4f}  (|Δ vs {GRU_REF_OVERALL}| = {abs(HR_cart_overall - GRU_REF_OVERALL):.4f})')
print(f'  Frenet  HR = {HR_fren_overall:.4f}  (|Δ vs {GRU_REF_OVERALL}| = {abs(HR_fren_overall - GRU_REF_OVERALL):.4f})')

if abs(HR_cart_overall - GRU_REF_OVERALL) < 0.005:
    pred_GRU_tr = pred_cart
    pred_GRU_ts = base_test + test_GRU
    gru_format = 'cart'
elif abs(HR_fren_overall - GRU_REF_OVERALL) < 0.005:
    pred_GRU_tr = pred_fren
    test_GRU_inv = (test_GRU[:, 0:1] * eL_ts + test_GRU[:, 1:2] * eN_ts + test_GRU[:, 2:3] * eB_ts)
    pred_GRU_ts = base_test + test_GRU_inv
    gru_format = 'frenet'
else:
    raise RuntimeError('GRU cache 형태 불명 — EXP #33 진행 불가')

err_GRU = np.linalg.norm(y_train - pred_GRU_tr, axis=1)
hit_GRU = err_GRU < HIT_RADIUS
HR_GRU = hr_triple(hit_GRU, minority_mask_tr)
print(f'\nGRU form={gru_format}  HR: overall={HR_GRU["overall"]:.4f}  major={HR_GRU["major"]:.4f}  minor={HR_GRU["minor"]:.4f}')

# K15/GRU 2-way oracle ceiling
choose_GRU = err_GRU < err_K15
pred_oracle_KG = np.where(choose_GRU[:, None], pred_GRU_tr, pred_K15_tr)
err_oracle_KG = np.linalg.norm(y_train - pred_oracle_KG, axis=1)
HR_oracle_KG = hr_triple(err_oracle_KG < HIT_RADIUS, minority_mask_tr)
print(f'\nK15/GRU 2-way oracle: overall={HR_oracle_KG["overall"]:.4f}  Δo +{HR_oracle_KG["overall"]-K15_REF_HR["overall"]:.4f}')


## 셀 6 — Phase 0: selector target + agreement + recovery_mask + #32 classifier 재현

`y_sel_kg = (err_GRU < err_K15)` — 1 = GRU wins.
**#32 classifier OOF/test 로드 + Soft baseline (T=1, Δ=∞, α=1) HR 비트 일치 sanity** = Phase A 핵심 가드.


In [ ]:
y_sel_kg = (err_GRU < err_K15).astype(np.int8)
n_GRUw = int(y_sel_kg.sum())
n_K15w = int((y_sel_kg == 0).sum())
maj_baseline_kg = max(n_GRUw, n_K15w) / len(y_sel_kg)
print(f'GRU wins: {n_GRUw} ({n_GRUw/len(y_sel_kg):.4f}) | K15 wins: {n_K15w} ({n_K15w/len(y_sel_kg):.4f})')

both_kg = hit_K15 & hit_GRU
K15_only_kg = hit_K15 & ~hit_GRU
GRU_only_kg = ~hit_K15 & hit_GRU
neither_kg = ~hit_K15 & ~hit_GRU
print(f'Agreement: both {int(both_kg.sum())} | K15_only {int(K15_only_kg.sum())} | GRU_only {int(GRU_only_kg.sum())} | neither {int(neither_kg.sum())}')

recovery_mask = K15_only_kg | GRU_only_kg
print(f'Recovery mask: {int(recovery_mask.sum())} sample ({recovery_mask.mean():.4f})')

# === #32 classifier OOF/test 로드 + Soft baseline 비트 일치 sanity ===
oof_clf32_per_seed  = [np.load(p) for p in CLF32_OOF_PATHS]
test_clf32_per_seed = [np.load(p) for p in CLF32_TEST_PATHS]
oof_clf32_mean  = np.mean(oof_clf32_per_seed, axis=0).astype(np.float32)
test_clf32_mean = np.mean(test_clf32_per_seed, axis=0).astype(np.float32)
assert oof_clf32_mean.shape == (10000,) and test_clf32_mean.shape == (10000,)
print(f'\n#32 classifier cache 로드: oof_mean shape {oof_clf32_mean.shape}, test_mean shape {test_clf32_mean.shape}')
print(f'  oof_clf32 분포: min={oof_clf32_mean.min():.4f}, p25={np.percentile(oof_clf32_mean,25):.4f}, median={np.median(oof_clf32_mean):.4f}, p75={np.percentile(oof_clf32_mean,75):.4f}, max={oof_clf32_mean.max():.4f}')
print(f'  test_clf32 분포: min={test_clf32_mean.min():.4f}, p25={np.percentile(test_clf32_mean,25):.4f}, median={np.median(test_clf32_mean):.4f}, p75={np.percentile(test_clf32_mean,75):.4f}, max={test_clf32_mean.max():.4f}')

# #32 classifier metrics (informational)
clf32_overall_auc = float(roc_auc_score(y_sel_kg, oof_clf32_mean))
clf32_recovery_auc = float(roc_auc_score(y_sel_kg[recovery_mask], oof_clf32_mean[recovery_mask]))
print(f'\n#32 classifier metrics (재현):  overall AUC = {clf32_overall_auc:.4f}  recovery AUC = {clf32_recovery_auc:.4f}')
print(f'  (#32 보고: recovery-AUC 0.5547)')

# Soft baseline (T=1, Δ=∞, α=1) reproduce
w_base = apply_soft_lever(oof_clf32_mean, *BASELINE_CELL)[:, None]
pred_soft_base_tr = w_base * pred_GRU_tr + (1 - w_base) * pred_K15_tr
HR_soft_base = hr_triple(np.linalg.norm(y_train - pred_soft_base_tr, axis=1) < HIT_RADIUS, minority_mask_tr)
print(f'\nSoft baseline (T=1, Δ=∞, α=1) 재현 HR: overall={HR_soft_base["overall"]:.4f}  major={HR_soft_base["major"]:.4f}  minor={HR_soft_base["minor"]:.4f}')
do_base = HR_soft_base['overall'] - K15_REF_HR['overall']
print(f'  Δo vs K15 = +{do_base:.4f}  (vs #32 보고 +{SOFT_BASELINE_DO:.4f})')
assert abs(HR_soft_base['overall'] - SOFT_BASELINE_HR_OVERALL) < 0.0005,     f'Soft baseline 재현 fail: {HR_soft_base["overall"]:.4f} vs ref {SOFT_BASELINE_HR_OVERALL:.4f}'
print('Soft baseline sanity OK — #32 cache 비트 일치 확인 → Phase A 진입 가능')


## 셀 7 — Phase A: 27 cell 3-D sweep + per-cell HR + gates

3-축 적용 순서: `p → σ scaling → max_w clip → centered shrinkage → final w`. 학습 0, OOF HR 계산만.

**가드 framework**:
- G1 (informational): √27·0.0014 ≈ +0.0073 (multi-comparison 부담 표시만)
- **Floor (primary)**: Δo > +0.0037 (Soft baseline 초과)
- **4th guard (primary)**: Δo > +0.0020 (per-cell LB자격)
- G2/G3: Δmajor > −0.002, Δminor > −0.005
- **채택 식**: Floor strict AND 4th guard strict AND G2 AND G3 → OOF Δo max 1개


In [ ]:
def evaluate_cell_gates(HR, floor_thr=FLOOR_PRIMARY):
    d_o = HR['overall'] - K15_REF_HR['overall']
    d_M = HR['major']   - K15_REF_HR['major']
    d_m = HR['minor']   - K15_REF_HR['minor']
    g1_info  = d_o > G1_INFO_THR
    g2       = d_M > -0.002
    g3       = d_m > -0.005
    g4       = d_o > G4_THR
    floor_ok = d_o > floor_thr
    lb_ok    = bool(g4 and g2 and g3 and floor_ok)
    return dict(d_overall=float(d_o), d_major=float(d_M), d_minor=float(d_m),
                G1_info=bool(g1_info), G2=bool(g2), G3=bool(g3),
                G4=bool(g4), Floor=bool(floor_ok), lb_eligible=lb_ok)


print(f'=== Phase A: 27 cell 3-D sweep ===\n')
cell_results = []  # list of dict: T, Delta, alpha, name, HR, gates
_t_pA = time.time()

for T, Delta, alpha in itertools.product(SIGMA_T_GRID, MAX_W_DELTA_GRID, ALPHA_GRID):
    w_cell = apply_soft_lever(oof_clf32_mean, T=T, Delta=Delta, alpha=alpha)[:, None]
    pred_cell = w_cell * pred_GRU_tr + (1 - w_cell) * pred_K15_tr
    err_cell = np.linalg.norm(y_train - pred_cell, axis=1)
    hit_cell = err_cell < HIT_RADIUS
    HR_cell = hr_triple(hit_cell, minority_mask_tr)
    gates_cell = evaluate_cell_gates(HR_cell)
    is_baseline = (T, Delta, alpha) == BASELINE_CELL
    cell_results.append(dict(
        T=T, Delta=Delta, alpha=alpha,
        name=cell_name(T, Delta, alpha),
        is_baseline=is_baseline,
        HR=HR_cell,
        **gates_cell,
        w_range=(float(w_cell.min()), float(w_cell.max())),
    ))

print(f'Phase A 계산 시간: {time.time()-_t_pA:.1f}s ({N_CELLS} cell)\n')

# Per-cell 표
print(f'{"#":>3} {"name":<18} {"T":>4} {"Δ":>5} {"α":>5} {"overall":>8} {"major":>8} {"minor":>8} {"Δo":>8} {"ΔM":>8} {"Δm":>8} {"G1i":>4} {"G2":>3} {"G3":>3} {"G4":>3} {"F":>3} {"LB":>3} {"base":>5}')
print('-' * 145)
for i, c in enumerate(cell_results):
    delta_str = '∞' if c['Delta'] == float('inf') else f'{c["Delta"]:.2f}'
    base_flag = '★' if c['is_baseline'] else ' '
    print(f'[{i+1:2d}] {c["name"]:<18} {c["T"]:>4.1f} {delta_str:>5} {c["alpha"]:>5.2f} '
          f'{c["HR"]["overall"]:>8.4f} {c["HR"]["major"]:>8.4f} {c["HR"]["minor"]:>8.4f} '
          f'{c["d_overall"]:>+8.4f} {c["d_major"]:>+8.4f} {c["d_minor"]:>+8.4f} '
          f'{"O" if c["G1_info"] else "X":>4} {"O" if c["G2"] else "X":>3} '
          f'{"O" if c["G3"] else "X":>3} {"O" if c["G4"] else "X":>3} '
          f'{"O" if c["Floor"] else "X":>3} {"O" if c["lb_eligible"] else "X":>3} {base_flag:>5}')

# Floor 통과 cell count + LB eligible count
floor_pass_count = sum(1 for c in cell_results if c['Floor'])
lb_eligible_count = sum(1 for c in cell_results if c['lb_eligible'])
print(f'\nFloor (Δo > +{FLOOR_PRIMARY:.4f}) 통과 cell: {floor_pass_count}/{N_CELLS}')
print(f'LB eligible (Floor + G4 + G2 + G3 strict) cell: {lb_eligible_count}/{N_CELLS}')
print(f'G1 informational (Δo > +{G1_INFO_THR:.4f}) 통과 cell: {sum(1 for c in cell_results if c["G1_info"])}/{N_CELLS}')


## 셀 8 — Phase A 추가 분석: marginal effect + 2-축 interaction heatmap + rank

(b) σ T marginal × 3, (c) max_w Δ marginal × 3, (d) α marginal × 3.
(e) 3 × interaction heatmap (σ×max_w, σ×α, max_w×α).
(f) top-5 + bot-5 rank.


In [ ]:
# Build a DataFrame for easy aggregation
df_cells = pd.DataFrame([dict(
    T=c['T'], Delta=c['Delta'], alpha=c['alpha'], name=c['name'],
    do=c['d_overall'], dM=c['d_major'], dm=c['d_minor'],
    overall=c['HR']['overall'], lb_eligible=c['lb_eligible'], floor=c['Floor'],
) for c in cell_results])

# Marginal effect (각 축 값마다 9 cell 평균)
print('=== (b/c/d) Marginal effect per axis (각 값마다 9 cell 평균 Δo) ===\n')
for axis, name, grid in [('T', 'σ (temperature T)', SIGMA_T_GRID),
                          ('Delta', 'max_w (cap Δ)', MAX_W_DELTA_GRID),
                          ('alpha', 'centered shrinkage α', ALPHA_GRID)]:
    print(f'{name}:')
    for val in grid:
        sub = df_cells[df_cells[axis] == val]
        val_str = '∞' if val == float('inf') else f'{val}'
        print(f'  {axis}={val_str:>4}: Δo mean = {sub["do"].mean():+.4f}  ΔM mean = {sub["dM"].mean():+.4f}  Δm mean = {sub["dm"].mean():+.4f}  '
              f'(floor 통과 {sub["floor"].sum()}/{len(sub)}, LB자격 {sub["lb_eligible"].sum()}/{len(sub)})')
    print()

# 2-축 interaction heatmap (한 축 평균하여 3×3 표)
print('=== (e) 2-축 interaction heatmap (Δo, 제3축 평균) ===\n')

def _delta_str(v):
    return '∞' if v == float('inf') else f'{v}'

for ax_a, ax_b, fixed_name, fixed_grid in [
    ('T', 'Delta', 'α', ALPHA_GRID),
    ('T', 'alpha', 'Δ', MAX_W_DELTA_GRID),
    ('Delta', 'alpha', 'σ T', SIGMA_T_GRID),
]:
    grid_a = SIGMA_T_GRID if ax_a == 'T' else (MAX_W_DELTA_GRID if ax_a == 'Delta' else ALPHA_GRID)
    grid_b = SIGMA_T_GRID if ax_b == 'T' else (MAX_W_DELTA_GRID if ax_b == 'Delta' else ALPHA_GRID)
    print(f'{ax_a} × {ax_b} (제3축 {fixed_name} 평균):')
    header = '     ' + '  '.join([f'{_delta_str(b):>8}' for b in grid_b])
    print(f'  {ax_b:>4} → ' + header)
    for a_val in grid_a:
        row_str = f'  {ax_a}={_delta_str(a_val):>4}    '
        for b_val in grid_b:
            sub = df_cells[(df_cells[ax_a] == a_val) & (df_cells[ax_b] == b_val)]
            mean_do = sub['do'].mean()
            row_str += f'  {mean_do:>+8.4f}'
        print(row_str)
    print()

# Top-5 + bot-5
print('=== (f) Rank: top-5 + bot-5 cell (OOF Δo) ===\n')
df_sorted = df_cells.sort_values('do', ascending=False).reset_index(drop=True)
print('Top-5 cell:')
for i in range(5):
    r = df_sorted.iloc[i]
    print(f'  [{i+1}] {r["name"]:<18}  Δo {r["do"]:+.4f}  ΔM {r["dM"]:+.4f}  Δm {r["dm"]:+.4f}  '
          f'(LB자격 {"O" if r["lb_eligible"] else "X"}, Floor {"O" if r["floor"] else "X"})')
print('\nBot-5 cell:')
for i in range(5):
    r = df_sorted.iloc[len(df_sorted) - 1 - i]
    print(f'  [{i+1}] {r["name"]:<18}  Δo {r["do"]:+.4f}  ΔM {r["dM"]:+.4f}  Δm {r["dm"]:+.4f}')

# Top-3 cell 식별 (Phase B 조건부 재측정용)
top3_cells = df_sorted.head(3)['name'].tolist()
print(f'\nTop-3 cells (Phase B 재측정 대상): {top3_cells}')


## 셀 9 — Phase B (조건부): classifier 강화 + top-3 cell 재측정

**진입 trigger**: Phase A floor 통과 cell = 0 (= sub-lever sweep만으로 baseline 못 이김).

3 classifier variant 순차 학습 (각 ~5분, 총 ~15분):
1. **V3 26** (26 feat)
2. **Residual-derived** (8 feat) — K15 OOF residual 3축+norm + GRU OOF residual 3축+norm (test 미사용, leakage 회피)
3. **Hybrid** (23 feat) — V3 15 + Residual-derived

가드: **recovery-AUC > 0.60 strict** (강화 검증). 미달 시 해당 variant 폐기.

best variant (recovery-AUC max + test 가용) → baseline + top-3 cell 재측정.


In [ ]:
phase_b_run = (floor_pass_count == 0)
phase_b_results = {}

if phase_b_run:
    print('=== Phase B: classifier 강화 진입 (Phase A floor 통과 cell = 0) ===\n')

    resid_K15 = (y_train - pred_K15_tr).astype(np.float32)
    resid_GRU = (y_train - pred_GRU_tr).astype(np.float32)
    norm_K15 = _norm(resid_K15)[:, None]
    norm_GRU = _norm(resid_GRU)[:, None]
    X_tr_resid = np.column_stack([resid_K15, norm_K15, resid_GRU, norm_GRU]).astype(np.float32)
    X_tr_hybrid = np.column_stack([X_tr_v15, X_tr_resid]).astype(np.float32)
    assert X_tr_resid.shape == (10000, 8) and X_tr_hybrid.shape == (10000, 23)

    variants = [
        ('v3_26',     X_tr_v26,    X_ts_v26, 'V3 26 feat'),
        ('resid',     X_tr_resid,  None,     'Residual-derived 8 feat (test 미사용)'),
        ('hybrid',    X_tr_hybrid, None,     'Hybrid V3 15 + Residual 23 feat (test 미사용)'),
    ]

    for v_name, X_tr_v, X_ts_v, desc in variants:
        print(f'--- {v_name}: {desc} ---')
        _t = time.time()
        res = run_classifier_seeds(X_tr=X_tr_v, y_target=y_sel_kg, X_ts=X_ts_v,
                                    stratify=minority_mask_tr.astype(int))
        mets = compute_classifier_metrics(res['oof_prob_mean'], y_sel_kg,
                                           recovery_mask=recovery_mask, per_seed=res['per_seed'])
        rec_pass = mets['recovery_auc'] > AUC_RECOVERY_STRONG_THR
        phase_b_results[v_name] = dict(
            result=res, metrics=mets, pass_recovery_strong=rec_pass,
            has_test=(X_ts_v is not None), time_sec=time.time() - _t,
        )
        print(f'  overall AUC={mets["overall_auc"]:.4f}  recovery-AUC={mets["recovery_auc"]:.4f}  '
              f'fold_std={mets["fold_std_mean"]:.4f}  → {"PASS O strong" if rec_pass else "FAIL X strong"}  '
              f'({time.time()-_t:.0f}s)\n')

    # best variant 선정 (recovery-AUC max + test 가용)
    test_variants = [(name, v) for name, v in phase_b_results.items() if v['has_test'] and v['pass_recovery_strong']]
    if test_variants:
        test_variants.sort(key=lambda x: -x[1]['metrics']['recovery_auc'])
        best_v_name, best_v_info = test_variants[0]
        print(f'\nbest variant (test 가용 + recovery-AUC > {AUC_RECOVERY_STRONG_THR}): {best_v_name}  '
              f'(recovery-AUC {best_v_info["metrics"]["recovery_auc"]:.4f})')
        oof_clf_strong = best_v_info['result']['oof_prob_mean']
        test_clf_strong = best_v_info['result']['test_prob_mean']

        # baseline + top-3 cell 재측정
        target_cells = [BASELINE_CELL] + [
            (c['T'], c['Delta'], c['alpha']) for c in cell_results if c['name'] in top3_cells
        ][:3]
        phase_b_cell_results = []
        print(f'\n--- Phase B 재측정: baseline + top-3 cell ({len(target_cells)} cell) ---')
        for T, Delta, alpha in target_cells:
            w_b = apply_soft_lever(oof_clf_strong, T=T, Delta=Delta, alpha=alpha)[:, None]
            pred_b = w_b * pred_GRU_tr + (1 - w_b) * pred_K15_tr
            HR_b = hr_triple(np.linalg.norm(y_train - pred_b, axis=1) < HIT_RADIUS, minority_mask_tr)
            gates_b = evaluate_cell_gates(HR_b)
            phase_b_cell_results.append(dict(
                T=T, Delta=Delta, alpha=alpha, name=cell_name(T, Delta, alpha),
                is_baseline=(T, Delta, alpha) == BASELINE_CELL, HR=HR_b, **gates_b,
            ))
            print(f'  {cell_name(T, Delta, alpha):<18}  Δo={gates_b["d_overall"]:+.4f}  '
                  f'ΔM={gates_b["d_major"]:+.4f}  Δm={gates_b["d_minor"]:+.4f}  '
                  f'F={"O" if gates_b["Floor"] else "X"} G4={"O" if gates_b["G4"] else "X"} '
                  f'LB={"O" if gates_b["lb_eligible"] else "X"}')

        # Save Phase B classifier cache (best variant)
        for i, seed in enumerate(SEEDS):
            np.save(f'results/soft_sweep_e33_clf_oof_seed{seed}.npy', best_v_info['result']['oof_probs_per_seed'][i])
            if best_v_info['has_test']:
                np.save(f'results/soft_sweep_e33_clf_test_seed{seed}.npy', best_v_info['result']['test_probs_per_seed'][i])
        np.save('results/soft_sweep_e33_clf_oof_mean.npy', oof_clf_strong)
        if best_v_info['has_test']:
            np.save('results/soft_sweep_e33_clf_test_mean.npy', test_clf_strong)
        print(f'\nPhase B {best_v_name} classifier cache 저장')
    else:
        print(f'\nPhase B: test 가용 + recovery-AUC > {AUC_RECOVERY_STRONG_THR} variant 0개')
        print(f'→ Phase B 폐기. Phase A 결과로 최종 결정 (LB skip 권장)')
        best_v_name = None
        oof_clf_strong = None
        test_clf_strong = None
        phase_b_cell_results = []
else:
    print(f'Phase A floor 통과 cell {floor_pass_count} ≥ 1 → Phase B skip')
    best_v_name = None
    oof_clf_strong = None
    test_clf_strong = None
    phase_b_cell_results = []


## 셀 10 — Phase 3: best cell 선택 + recall analysis

**채택 식**: Floor strict AND 4th guard strict AND G2 AND G3 통과 cell 중 OOF Δo max 1개.

Phase A pool + (조건부) Phase B 재측정 pool에서 통합 선정.


In [ ]:
# Phase A LB eligible cells
phase_a_eligible = [c for c in cell_results if c['lb_eligible']]
phase_b_eligible = [c for c in phase_b_cell_results if c['lb_eligible']] if phase_b_run else []

print(f'=== Phase 3: best cell 선택 ===')
print(f'Phase A LB eligible: {len(phase_a_eligible)} cell')
print(f'Phase B LB eligible: {len(phase_b_eligible)} cell (Phase B run = {phase_b_run})')

candidates = []
for c in phase_a_eligible:
    candidates.append(dict(source='phaseA', **c))
for c in phase_b_eligible:
    candidates.append(dict(source=f'phaseB_{best_v_name}', **c))

if not candidates:
    print(f'\n⚠ LB eligible cell 0개 — LB 슬롯 사용 권장 안 함')
    print(f'  Floor +{FLOOR_PRIMARY:.4f} strict + 4th guard +{G4_THR:.4f} strict + G2 + G3 통과 cell 없음')
    best_cell = None
    best_source = None
else:
    candidates.sort(key=lambda x: -x['d_overall'])
    best_cell = candidates[0]
    best_source = best_cell['source']
    print(f'\nbest cell ({len(candidates)} eligible 중 OOF Δo max):')
    print(f'  source: {best_source}')
    print(f'  name:   {best_cell["name"]}  (T={best_cell["T"]}, Δ={best_cell["Delta"]}, α={best_cell["alpha"]})')
    print(f'  Δo:     {best_cell["d_overall"]:+.4f}  ΔM: {best_cell["d_major"]:+.4f}  Δm: {best_cell["d_minor"]:+.4f}')
    print(f'  vs #32 LB margin: {best_cell["d_overall"] - SOFT_BASELINE_DO:+.4f} (OOF)')

# Recall analysis (best cell만)
print(f'\n=== Recall analysis ===')
print(f'K15/GRU recovery: K15_only {int(K15_only_kg.sum())} / GRU_only {int(GRU_only_kg.sum())}')


def _recall_cell(w_arr):
    use_GRU = w_arr > 0.5
    GRU_only_recovered = int((use_GRU & GRU_only_kg).sum())
    K15_only_lost = int((use_GRU & K15_only_kg).sum())
    return dict(GRU_selected=int(use_GRU.sum()),
                GRU_only_recovered=GRU_only_recovered,
                K15_only_lost=K15_only_lost,
                net_gain_hit=GRU_only_recovered - K15_only_lost)


# baseline + best cell recall (Phase A oof_clf32_mean 기준)
print(f'\nBaseline (T=1, Δ=∞, α=1, raw p_clf):')
w_base_flat = apply_soft_lever(oof_clf32_mean, *BASELINE_CELL)
print(f'  {_recall_cell(w_base_flat)}')
if best_cell is not None:
    src = oof_clf_strong if best_source.startswith('phaseB') else oof_clf32_mean
    w_best = apply_soft_lever(src, T=best_cell['T'], Delta=best_cell['Delta'], alpha=best_cell['alpha'])
    print(f'\nBest cell ({best_cell["name"]}):')
    print(f'  {_recall_cell(w_best)}')


## 셀 11 — Submission 생성 (best 1개만)

**plan §4.18 + memory `feedback-submission-best-only` (2026-05-28 정책 반전)**: Phase 3 선정 best cell **1개만** 생성. 27 cell HR/가드 정량 자산은 `results/soft_sweep_e33_meta.json`에 전부 보존되므로 sweep 자체가 손실되지 않음.

best_cell None (LB eligible 0) 시 submission 0개 → 콘솔에 "LB 슬롯 사용 권장 안 함" 명시 + 이전 EXP LB 유지.

명명: `submission_split_e33_{best_cell_name}_ms3.csv` (예: `T15_DINF_a100` = T=1.5, Δ=∞, α=1.0)


In [ ]:
def make_submission(pred_test_abs, name):
    assert pred_test_abs.shape == (10000, 3)
    assert np.isfinite(pred_test_abs).all()
    df = pd.DataFrame({
        'id': test_ids,
        'x': pred_test_abs[:, 0],
        'y': pred_test_abs[:, 1],
        'z': pred_test_abs[:, 2],
    })
    df.to_csv(name, index=False)
    return name


print('=== Submission 생성 (best 1개만, memory feedback-submission-best-only) ===\n')
sub_paths = {}

if best_cell is None:
    print(f'best_cell None (LB eligible cell 0개)')
    print(f'→ submission 0개 생성. LB 슬롯 사용 권장 안 함.')
    print(f'→ #32 Soft baseline LB 0.6910 유지가 최종.')
    chosen_lb_path = None
else:
    # best_cell 출처 (phaseA classifier OOF=oof_clf32_mean, phaseB classifier=oof_clf_strong)
    test_prob_src = test_clf_strong if best_source.startswith('phaseB') else test_clf32_mean
    w_test_best = apply_soft_lever(
        test_prob_src,
        T=best_cell['T'], Delta=best_cell['Delta'], alpha=best_cell['alpha'],
    )[:, None]
    pred_test_best = w_test_best * pred_GRU_ts + (1 - w_test_best) * pred_K15_ts

    if best_source == 'phaseA':
        chosen_lb_path = f'submission_split_e33_{best_cell["name"]}_ms3.csv'
    else:
        chosen_lb_path = f'submission_split_e33_phaseB_{best_v_name}_{best_cell["name"]}_ms3.csv'

    make_submission(pred_test_best, chosen_lb_path)
    sub_paths[best_cell['name']] = chosen_lb_path

    print(f'  best cell: {best_cell["name"]}  source: {best_source}')
    print(f'  cell params: T={best_cell["T"]}, Δ={best_cell["Delta"]}, α={best_cell["alpha"]}')
    print(f'  Δo OOF: {best_cell["d_overall"]:+.4f}  margin vs #32 baseline: {best_cell["d_overall"]-SOFT_BASELINE_DO:+.4f}')
    print(f'  → submission: {chosen_lb_path}')
    print(f'  → LB 슬롯 1개 권장')


## 셀 12 — Meta 저장 + Drive 복사 + files.download (best 1개만, memory feedback-submission-best-only)

In [ ]:
def safe(x):
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, np.ndarray): return x.tolist()
    if x == float('inf'): return 'inf'
    return x


def _cell_to_dict(c):
    d = dict(c)
    if d.get('Delta') == float('inf'):
        d['Delta'] = 'inf'
    return d


meta = {
    'protocol': 'EXP #33 L17 K15/GRU Soft 고도화 (3-D sub-lever sweep + 조건부 classifier 강화)',
    'seeds': SEEDS, 'n_folds': N_FOLDS,
    'minority_threshold': MINORITY_THRESHOLD,
    'baseline_cell': dict(T=BASELINE_CELL[0], Delta='inf', alpha=BASELINE_CELL[2]),
    'grid': {
        'sigma_T': SIGMA_T_GRID,
        'max_w_Delta': [d if d != float('inf') else 'inf' for d in MAX_W_DELTA_GRID],
        'alpha': ALPHA_GRID,
        'n_cells': N_CELLS,
    },
    'thresholds': {
        'g1_informational': G1_INFO_THR,
        'g4_primary_per_cell': G4_THR,
        'floor_primary_soft_baseline': FLOOR_PRIMARY,
        'soft_baseline_do_ref': SOFT_BASELINE_DO,
        'minorgru_floor_ref': MINORGRU_FLOOR_DO,
        'combined_std': COMBINED_STD,
        'auc_recovery_strong': AUC_RECOVERY_STRONG_THR,
    },
    'k15_ref_HR': K15_REF_HR,
    'gru_format': gru_format,
    'gru_HR': HR_GRU,
    'oracle_HR_K15GRU': HR_oracle_KG,
    'ceiling_gain_K15GRU': CEILING_GAIN_KG,
    'agreement_K15_GRU': dict(both=int(both_kg.sum()), K15_only=int(K15_only_kg.sum()),
                                GRU_only=int(GRU_only_kg.sum()), neither=int(neither_kg.sum())),
    'recovery_mask_count': int(recovery_mask.sum()),
    'soft_baseline_reproduce_HR': HR_soft_base,
    'clf32_metrics_reproduce': dict(overall_auc=clf32_overall_auc, recovery_auc=clf32_recovery_auc),
    'phase_a': {
        'n_cells': N_CELLS,
        'cell_results': [_cell_to_dict(c) for c in cell_results],
        'floor_pass_count': floor_pass_count,
        'lb_eligible_count': lb_eligible_count,
        'top3_cells': top3_cells,
    },
    'phase_b_run': bool(phase_b_run),
    'phase_b_results': {
        name: {
            'per_seed': info['result']['per_seed'],
            'metrics': info['metrics'],
            'pass_recovery_strong': bool(info['pass_recovery_strong']),
            'has_test': bool(info['has_test']),
            'time_sec': info['time_sec'],
        } for name, info in phase_b_results.items()
    } if phase_b_run else {},
    'phase_b_best_variant': best_v_name,
    'phase_b_cell_results': [_cell_to_dict(c) for c in phase_b_cell_results],
    'best_cell': _cell_to_dict(best_cell) if best_cell is not None else None,
    'best_source': best_source,
    'lb_chosen_path': chosen_lb_path,
    'submission_best_only': sub_paths,  # 0개 (best_cell None) or 1개
    'submission_policy': 'best_only (memory feedback-submission-best-only, 2026-05-28 정책 반전)',
    'minority_train_count': int(minority_mask_tr.sum()),
    'minority_test_count': int(minority_mask_ts.sum()),
}
with open('results/soft_sweep_e33_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=safe)
print('results/soft_sweep_e33_meta.json 저장 (27 cell HR + 가드 정량 자산 전부 보존)')

# Drive 복사 + files.download (best 1개만, memory feedback-submission-best-only)
try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive'
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/* {results_drive}/
    if os.path.exists('hr_aware_soft_sweep_e33.ipynb'):
        !cp hr_aware_soft_sweep_e33.ipynb {DRIVE_BASE}/
    if sub_paths:
        for p in sub_paths.values():
            !cp {p} {DRIVE_BASE}/
            files.download(p)
        print(f'Drive 복사 + 다운로드 완료 (best submission 1개: {list(sub_paths.values())[0]})')
    else:
        print('best_cell None → submission 0개. Drive 복사 skip (meta.json + ipynb만 복사)')
except ImportError:
    print('Colab 아님 — Drive 복사 skip')
